# Lección 3: Semana 3, cuaderno de grupo

Cuaderno de grupo de la semana 3. Comunidades de Louvain en la red de enlaces y una etiqueta dada con una red de una capa oculta, cada una comparada con la clase mayoritaria. La lista de inicio se escribe en el cuaderno del proyecto.

---
<font size="3">
Christian Cabrera Jojoa<br>
Assistant Research Professor<br>
Department of Computer Science and Technology<br>
University of Cambridge<br>
chc79@cam.ac.uk
</font>

---
**Curso:** Data Mining<br>
**Departamento del curso:** Departamento de Matemáticas y Estadística - Facultad de Ciencias Exactas y Naturales<br>
**Institución del curso:** Universidad de Nariño


## Instrucciones

**Propósito.** Cuaderno de grupo de la semana 3 (Lecciones 5 y 6). Su grupo sigue como equipo interno de una alcaldía, con los artículos de su secretaría. El producto del proyecto es una lista de artículos para los nuevos integrantes del equipo.

**Esta semana.** El grupo usa un método de la Lección 5 y un método de la Lección 6: las comunidades de Louvain y una red con una capa oculta.
Louvain, porque agrupa artículos que se mencionan más entre ellos que con el resto, y da un identificador a cada artículo, también a los aislados (cada uno queda en su propia comunidad de tamaño 1).
La red de una capa oculta, porque es el segundo método de la Lección 6 y se compara, como el otro, con la clase mayoritaria.
Cada método se compara con la clase mayoritaria. Louvain no usa la etiqueta para armar las comunidades. Para compararlo con esa línea base, cada comunidad hereda la clase más frecuente de sus artículos de **entrenamiento** y esa clase se aplica a los artículos apartados de la misma comunidad. Si una comunidad no tiene artículos de entrenamiento, se usa la clase mayoritaria del entrenamiento.
Los resultados son `outputs/grafo.csv` y `outputs/prediccion.csv`. El cuaderno del proyecto los lee cuando arma la lista de inicio.

**Antes de empezar.** Ejecuten primero los cuadernos individuales de las Lecciones 5 y 6. Este cuaderno lo escribe el grupo. Traigan de los laboratorios los parámetros y las funciones.

**Datos de entrada.** El archivo de su secretaría: G1 Agricultura, G2 Salud mental, G3 Medio ambiente, G4 Inclusión Social y G5 Vivienda. La columna `links` trae los enlaces de la descripción a otros artículos del mismo archivo. Un enlace es una mención en el texto, no una cita. Las categorías las pusieron quienes editan Wikipedia.

**Materiales.** [L5 Grafos](https://colab.research.google.com/github/cabrerac/cabrerac.github.io/blob/gh-pages/assets/notebooks/26-udenar-data-mining/l5-graphs.ipynb) y [L6 Predicción](https://colab.research.google.com/github/cabrerac/cabrerac.github.io/blob/gh-pages/assets/notebooks/26-udenar-data-mining/l6-prediction.ipynb).

**Las decisiones son del grupo.** El grupo puede cambiar la pregunta o el alcance. Escriban la razón en el paso donde lo cambia.

**Pregunta del grupo.** ¿Qué comunidades de menciones hay en su secretaría, y la predicción de la red con una capa oculta supera a la clase mayoritaria?

**Mapa del cuaderno**

| Ejercicio | Tema | Entregable clave |
|-----------|------|------------------|
| **0** | Configuración | Grupo, etiqueta, datos, parámetros y funciones |
| **1** | Comprensión del negocio | La nota de si la pregunta cambió, y qué esperan del grafo |
| **2** | Comprensión de los datos | Títulos únicos, enlaces y la etiqueta |
| **3** | Preparación de los datos | Los enlaces, el grafo sin dirección, el conjunto apartado y la preparación |
| **4** | Modelado | Louvain, la regla de mayoría por comunidad y la red con una capa oculta |
| **5** | Evaluación | Un artículo por su comunidad, aciertos y errores, si cada método cuenta, `outputs/grafo.csv` y `outputs/prediccion.csv` |
| **6** | Despliegue | Los artículos para leer (hasta 30) y la interpretación |

**Cómo leer cada paso.** Cada paso empieza con **En este paso**. Si el paso tiene código, termina con una celda **Comprobar**.

---

## Ejercicio 0. Configuración

### Paso 0.1. Grupo, etiqueta y datos

En este paso eligen el grupo y descargan el archivo de su secretaría.
Cambien solo `GROUP`.
La etiqueta de la Lección 6 es `LABEL = "categoria_objetivo"`.
Vale 1 si la categoría del artículo (`categoria`) está en `TARGETS[GROUP]`, y 0 si no.

In [ ]:
from pathlib import Path  # Path arma las rutas de las carpetas

import pandas as pd  # pandas lee y guarda las tablas
import requests  # requests descarga el archivo de la secretaría

GROUP = "G1"  # cambie a G2, G3, G4 o G5
LABEL = "categoria_objetivo"  # 1 si categoria está en TARGETS[GROUP], 0 si no
TARGETS = {  # las categorías de la etiqueta de cada secretaría
    "G1": ["Agrarismo", "Organizaciones agrícolas"],  # Agricultura
    "G2": ["Trastornos mentales", "Psiquiatría"],  # Salud mental
    "G3": ["Conservacionismo"],  # Medio ambiente
    "G4": ["Hambrunas", "Hambre", "Bancos de alimentos"],  # Inclusión Social
    "G5": ["Hipotecas", "Arrendamiento", "Huelgas de alquileres", "Vivienda asequible", "Vivienda pública"],  # Vivienda
}
EXTRACTS = {  # el nombre y el archivo de cada secretaría
    "G1": ("Agricultura", "agricultura.csv"),  # G1
    "G2": ("Salud mental", "salud-mental.csv"),  # G2
    "G3": ("Medio ambiente", "medio-ambiente.csv"),  # G3
    "G4": ("Inclusión Social", "inclusion-social.csv"),  # G4
    "G5": ("Vivienda", "vivienda.csv"),  # G5
}
SECRETARIAT, FILENAME = EXTRACTS[GROUP]  # tomamos la secretaría y el archivo del grupo
DATA_DIR = Path("data")  # aquí queda el CSV
OUTPUTS_DIR = Path("outputs")  # aquí quedan grafo.csv y prediccion.csv
DATA_DIR.mkdir(parents=True, exist_ok=True)  # mkdir crea la carpeta de datos si no existe
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)  # mkdir crea la carpeta de salidas si no existe
path = DATA_DIR / FILENAME  # la ruta local del archivo
if not path.is_file():  # usa el archivo si ya está
    path.write_bytes(requests.get(f"https://cabrerac.github.io/assets/data/26-udenar-data-mining/{FILENAME}", timeout=60).content)  # requests.get descarga el archivo y write_bytes lo guarda
df = pd.read_csv(path, keep_default_na=False).drop_duplicates("title").reset_index(drop=True)  # read_csv lee los artículos y drop_duplicates deja una fila por título
df = df.rename(columns={"label": "categoria"})  # en el extracto la columna se llama label
print(f"Grupo {GROUP}: secretaría de {SECRETARIAT}, {len(df)} artículos sin títulos repetidos. Etiqueta: {LABEL}")  # print muestra el grupo y el tamaño

### Paso 0.2. Importaciones, parámetros y funciones de apoyo

En este paso preparan lo que usa el resto del cuaderno. Va en tres celdas.
Primero las importaciones de las Lecciones 5 y 6 que usa este cuaderno: `math`, `networkx`, `numpy` y las de scikit-learn, incluida `MLPClassifier`.

In [ ]:
# SU CÓDIGO. Paso 0.2a: las importaciones de las Lecciones 5 y 6

Después los parámetros de las Lecciones 5 y 6, con los valores de los laboratorios.

In [ ]:
# SU CÓDIGO. Paso 0.2b: los parámetros de las Lecciones 5 y 6, MAX_PAPERS y folds

Al final las funciones de tamaño: `support_floor` y `n_flagged`.

In [ ]:
# SU CÓDIGO. Paso 0.2c: support_floor y n_flagged

**Listo cuando:**
- Los parámetros tienen los valores de los laboratorios: `SEED`, `MARGIN`, `SPLIT_SEEDS`, `N_FOLDS`, `HIDDEN_UNITS`, `PENALTY` y los de las filas apartadas. También `MAX_PAPERS = 30`.
- `folds` es la validación cruzada en cinco pliegues de la Lección 6, con `SEED`.

**Comprobar:**

Comprobamos algunos valores de los laboratorios y las dos funciones con unos tamaños conocidos.

In [ ]:
assert callable(MLPClassifier) and callable(nx.community.louvain_communities) and folds.get_n_splits() == N_FOLDS  # las importaciones y los pliegues están
assert (SEED, HIDDEN_UNITS, PENALTY, MARGIN, SPLIT_SEEDS, MAX_PAPERS) == (7, 8, 1.0, 0.02, (7, 8, 9, 10, 11), 30)  # algunos valores de los laboratorios
assert support_floor(300) == 9 and n_flagged(300) == 60  # las funciones dan los tamaños de las lecciones
print("Paso 0.2: OK")  # print confirma la comprobación

---

## Ejercicio 1. Comprensión del negocio

### Paso 1.1. ¿Cambió la pregunta?

La pregunta y el problema están en el documento de requerimientos.
En este paso escriben solo si algo cambió.
Si la pregunta o el alcance siguen iguales, escriban "Sin cambio".
Si cambiaron, escriban la línea nueva y por qué.

**Cambio**

**ESCRIBA SU TEXTO AQUÍ**

### Paso 1.2. Qué esperan del grafo

En este paso escriben qué esperan del grafo, antes de mirar resultados.
¿Qué artículos creen que van a quedar en la misma comunidad de menciones?
¿Eso les sirve a una persona nueva? ¿Por qué?

**Lo que esperamos**

**ESCRIBA SU TEXTO AQUÍ**

---

## Ejercicio 2. Comprensión de los datos

### Paso 2.1. Títulos y enlaces

En este paso revisan los títulos y cómo vienen los enlaces, como en la Lección 5, Parte 2.
Aquí `df` ya viene sin títulos repetidos.
`listed` guarda, por artículo, la lista de títulos que nombra en `links`.

In [ ]:
# SU CÓDIGO. Paso 2.1: titles, N y listed

**Comprobar:**

Comprobamos que hay un título y una lista de enlaces por artículo.

In [ ]:
assert len(titles) == N == len(listed) == len(df)  # un título y una lista por artículo
print("Paso 2.1: OK")  # print confirma la comprobación

### Paso 2.2. La etiqueta y los atributos

En este paso arman la tabla de atributos y la etiqueta.
`attributes` usa el título como índice.
Lleva `kind`, `year`, `has_date`, `has_country`, `has_image` y `n_claims_without`.
`kind` deja como `Other` los tipos con menos de `support_floor(N)` artículos.
`y` vale 1 si `categoria` está en `TARGETS[GROUP]`, y 0 si no.
Impriman cuántos artículos tienen 1 y cuántos tienen 0.

In [ ]:
# SU CÓDIGO. Paso 2.2: attributes y y

**Comprobar:**

Comprobamos que hay una etiqueta por artículo y que solo vale 0 o 1.

In [ ]:
assert len(y) == N and set(y.unique()) <= {0, 1}  # una etiqueta por artículo, con 0 o 1
assert list(y.index) == list(attributes.index)  # la etiqueta va en el mismo orden que los atributos
print(f"Paso 2.2: OK. Etiqueta en 1: {int(y.sum())} de {N}")  # print confirma la comprobación

---

## Ejercicio 3. Preparación de los datos

### Paso 3.1. Los enlaces y el grafo sin dirección

En este paso arman los enlaces dentro del extracto, como en la Lección 5, Parte 3.
`links_out` va con dirección, por si quieren contar menciones de salida.
`neighbours` va sin dirección.
`graph` es el grafo sin dirección que usa Louvain.
Agreguen los nodos en el orden de `titles` y los vecinos con `sorted`, para que Louvain se repita de una sesión a otra.

In [ ]:
# SU CÓDIGO. Paso 3.1: links_out, neighbours y graph

**Comprobar:**

Comprobamos que el grafo tiene un nodo por artículo y que ningún artículo se enlaza a sí mismo.

In [ ]:
assert graph.number_of_nodes() == N == len(neighbours)  # un nodo por artículo
assert nx.number_of_selfloops(graph) == 0  # nadie se enlaza a sí mismo
print(f"Paso 3.1: OK. Enlaces sin dirección: {graph.number_of_edges()}")  # print confirma la comprobación

### Paso 3.2. El conjunto apartado

En este paso apartan `n_flagged(N)` filas con `SEED`, con la misma proporción de cada clase, como en la Lección 6.
Ningún modelo las ve al aprender.
Louvain arma las comunidades con todo el grafo, porque los enlaces se observan. La etiqueta de las filas apartadas no entra en la regla de mayoría por comunidad.

In [ ]:
# SU CÓDIGO. Paso 3.2: train, test, y_train y y_test

**Comprobar:**

Comprobamos el tamaño del conjunto apartado y que no comparte filas con el entrenamiento.

In [ ]:
assert len(test) == n_flagged(N) and len(train) + len(test) == N  # tamaño correcto
assert not set(train.index) & set(test.index)  # sin filas compartidas
print("Paso 3.2: OK")  # print confirma la comprobación

### Paso 3.3. La preparación

En este paso arman `prepare`, el `ColumnTransformer` de las diapositivas de la Lección 6.
En los números usen `add_indicator=False`, porque `has_date` ya marca el año faltante.

In [ ]:
# SU CÓDIGO. Paso 3.3: NUMBER_COLUMNS, CATEGORY_COLUMNS y prepare

**Comprobar:**

Comprobamos que la preparación usa las columnas de la tabla.

In [ ]:
assert set(NUMBER_COLUMNS + CATEGORY_COLUMNS) <= set(attributes.columns) and "kind" in CATEGORY_COLUMNS  # las columnas existen
print("Paso 3.3: OK")  # print confirma la comprobación

---

## Ejercicio 4. Modelado

### Paso 4.1. Comunidades de Louvain

En este paso agrupan los artículos con `nx.community.louvain_communities(graph, seed=SEED)`, como en la Lección 5, Louvain.
Calculen la modularidad con `nx.community.modularity`.
`community_of` guarda el identificador de la comunidad de cada artículo.
`community_size` guarda el tamaño de esa comunidad.
Impriman el número de comunidades, la modularidad y los tamaños de las más grandes.

In [ ]:
# SU CÓDIGO. Paso 4.1: communities, modularity, community_of y community_size

**Comprobar:**

Comprobamos que hay una comunidad por artículo y que la modularidad queda en su rango.

In [ ]:
assert len(community_of) == N == len(community_size)  # un identificador y un tamaño por artículo
assert 0 <= modularity <= 1  # la modularidad queda entre 0 y 1
print(f"Paso 4.1: OK. Comunidades: {len(communities)}. Modularidad: {modularity:.3f}")  # print confirma la comprobación

### Paso 4.2. La mayoría de cada comunidad

En este paso convierten Louvain en una predicción, para compararlo con la clase mayoritaria.
Para cada comunidad, la clase predicha es la más frecuente entre sus artículos de **entrenamiento**.
Si hay empate, o si la comunidad no tiene artículos de entrenamiento, usan la clase mayoritaria del entrenamiento.
`community_prediction` es esa clase en las filas apartadas, en el mismo orden que `test`.

In [ ]:
# SU CÓDIGO. Paso 4.2: community_prediction

**Comprobar:**

Comprobamos que hay una predicción por fila apartada y que solo vale 0 o 1.

In [ ]:
assert len(community_prediction) == len(test)  # una predicción por fila apartada
assert set(community_prediction).issubset({0, 1})  # solo vale 0 o 1
print("Paso 4.2: OK")  # print confirma la comprobación

### Paso 4.3. Una capa oculta

En este paso entrenan la red de una capa oculta de la Lección 6, con la preparación adelante: `make_pipeline(prepare, MLPClassifier(...))`.
Usen `hidden_layer_sizes=(HIDDEN_UNITS,)`, `alpha=PENALTY`, `max_iter=3000` y `random_state=SEED`.
`test_prediction` es lo que dice la red en las filas apartadas.
`train_prediction` es lo que dice en las filas de entrenamiento, con cinco pliegues (`cross_val_predict` y `folds`), así un modelo que no vio la fila la predice.

In [ ]:
# SU CÓDIGO. Paso 4.3: network, test_prediction y train_prediction

**Comprobar:**

Comprobamos que hay una predicción por fila apartada y por fila de entrenamiento.

In [ ]:
assert len(test_prediction) == len(test) and len(train_prediction) == len(train)  # una predicción por fila
print("Paso 4.3: OK")  # print confirma la comprobación

---

## Ejercicio 5. Evaluación

### Paso 5.1. Un artículo descrito por su comunidad

En este paso eligen un artículo y lo describen por su comunidad, como en la Lección 5, Parte 5.
`MY_ARTICLE` es un título del extracto que no está aislado.
Impriman su comunidad, el tamaño de esa comunidad y la modularidad del grafo.

In [ ]:
# SU CÓDIGO. Paso 5.1: MY_ARTICLE

**Comprobar:**

Comprobamos que el artículo está en el extracto y no está aislado.

In [ ]:
assert MY_ARTICLE in titles and community_size[MY_ARTICLE] > 1  # un título del extracto que no está aislado
print("Paso 5.1: OK")  # print confirma la comprobación

¿Qué dice su comunidad?
¿Los artículos que la integran se parecen en algo que una persona nueva reconocería?

**ESCRIBA SU TEXTO AQUÍ**

### Paso 5.2. Aciertos y errores en las filas apartadas

En este paso cuentan aciertos y errores de Louvain y de la red en las filas apartadas, como en la Lección 6, Parte 5.
Sumen la fila de la clase mayoritaria: dar a todos la etiqueta más frecuente del entrenamiento.
Es la línea de referencia.
`results` tiene una fila por método, con `TP`, `FP`, `FN`, `TN` y `exactitud`.

In [ ]:
# SU CÓDIGO. Paso 5.2: majority y results

**Comprobar:**

Comprobamos las tres filas y que los conteos suman las filas apartadas.

In [ ]:
assert list(results.index) == ["clase mayoritaria", "Louvain", "una capa oculta"]  # las tres filas, en este orden
assert (results[["TP", "FP", "FN", "TN"]].sum(axis=1) == len(test)).all()  # los conteos suman las filas apartadas
print("Paso 5.2: OK")  # print confirma la comprobación

### Paso 5.3. ¿Cuenta cada método?

En este paso deciden si cada método cuenta, como en la Lección 6, Parte 5.
`l6_counts` es `True` solo si la red acierta en las filas apartadas más que la clase mayoritaria, por un margen de `MARGIN` (0,02).
`louvain_counts` es `True` solo si la regla de mayoría por comunidad supera ese mismo margen.
Ese margen es una o dos filas de unas 60.
`seeds_holding` cuenta en cuántas de las 5 particiones de `SPLIT_SEEDS` se sostiene la red, con `clone`.
`louvain_seeds_holding` hace lo mismo con la regla de mayoría por comunidad. Las comunidades no se vuelven a calcular: solo cambian las filas de entrenamiento que dan la mayoría de cada comunidad.
Si `l6_counts` es `False`, la predicción de la red no apoya su lista. Es un resultado normal y válido, no un error.

In [ ]:
# SU CÓDIGO. Paso 5.3: l6_counts, louvain_counts, seeds_holding y louvain_seeds_holding

**Comprobar:**

Comprobamos que las dos decisiones son verdadero o falso y que las dos cuentas van de 0 a 5.

In [ ]:
assert isinstance(l6_counts, bool) and isinstance(louvain_counts, bool)  # las dos decisiones son verdadero o falso
assert 0 <= seeds_holding <= len(SPLIT_SEEDS) and 0 <= louvain_seeds_holding <= len(SPLIT_SEEDS)  # las dos cuentas van de 0 a 5
print(f"Paso 5.3: OK. l6_counts = {l6_counts} ({seeds_holding} de {len(SPLIT_SEEDS)}). louvain_counts = {louvain_counts} ({louvain_seeds_holding} de {len(SPLIT_SEEDS)})")  # print confirma la comprobación

### Paso 5.4. Guardar el grafo y la predicción

En este paso guardan los dos resultados de esta semana.

In [ ]:
# SU CÓDIGO. Paso 5.4: outputs/grafo.csv y outputs/prediccion.csv

**Listo cuando:**
- `grafo.csv` tiene `title`, `comunidad`, `tamano_comunidad`, `modularidad`, `aislado` y `graph_support`. `graph_support` es verdadero si la comunidad del artículo tiene al menos dos artículos de entrenamiento y su clase más frecuente en el entrenamiento es 1.
- `prediccion.csv` tiene `title`, `etiqueta`, `apartada`, `clase_predicha` y `prediction`. `clase_predicha` es la clase que dice la red. `prediction` es verdadero si esa clase es 1.

**Comprobar:**

Comprobamos las columnas de los dos archivos.

In [ ]:
grafo, prediccion = pd.read_csv(OUTPUTS_DIR / "grafo.csv"), pd.read_csv(OUTPUTS_DIR / "prediccion.csv")  # read_csv vuelve a leer los dos archivos
assert {"title", "comunidad", "tamano_comunidad", "modularidad", "aislado", "graph_support"} <= set(grafo.columns)  # están las columnas del grafo
assert {"title", "etiqueta", "apartada", "clase_predicha", "prediction"} <= set(prediccion.columns)  # están las columnas de la predicción
print("Paso 5.4: OK")  # print confirma la comprobación

---

## Ejercicio 6. Despliegue

### Paso 6.1. Artículos para leer

En este paso arman la lista de lectura de esta semana con los dos métodos.
Louvain filtra: solo entran los artículos con `graph_support`.
Si la predicción de la red cuenta (`l6_counts`), primero van los que la red marca con `prediction`.
Después ordenan por tamaño de comunidad, de mayor a menor, y por título.
Si la predicción no cuenta, ordenan solo por tamaño de comunidad y por título.
Guarden como máximo 30 artículos en `reading`, con `title`, `comunidad`, `tamano_comunidad` y `prediction`.
Esta lista es la lectura de los resultados de esta semana. La lista de inicio se escribe en el cuaderno del proyecto.

In [ ]:
# SU CÓDIGO. Paso 6.1: reading

### Paso 6.2. Interpretación de resultados

¿Por qué se seleccionaron los artículos de la lista resultante esta semana?
Explique la selección de acuerdo con las comunidades de Louvain, la predicción y la pregunta.
¿Cada método supera a la clase mayoritaria?
Si un método no la supera, escríbalo así y diga qué hecho lo cambiaría.

**Interpretación**

**ESCRIBA SU TEXTO AQUÍ**

**Comprobar:**

Comprobamos el largo de la lista y que todo viene de `graph_support`.

In [ ]:
n_support = int(grafo["graph_support"].sum())  # sum cuenta los artículos con graph_support
assert len(reading) == min(MAX_PAPERS, n_support)  # la lista no pasa de 30
assert set(reading["title"]) <= set(grafo.loc[grafo["graph_support"], "title"])  # todo viene de graph_support
print("Paso 6.2: OK. Armen el ZIP.")  # print confirma la comprobación

---

## Registro de contribución

En este paso cada integrante anota su tarea concreta y el grupo dice quién consolida el ZIP.

**ESCRIBA SU TEXTO AQUÍ**

## Para la reflexión individual

Use la [plantilla de la reflexión de la semana 3](https://cabrerac.github.io/assets/documents/26-udenar-data-mining/reflection-week-3-template.docx) y mantenga sus títulos.
Extensión: entre 700 y 1000 palabras.
Por debajo de 500 palabras el texto queda corto.

## Entrega

El ZIP se entrega el martes 1 dic 2026, 23:59 (hora de Colombia), con el cuaderno ejecutado y renombrado.
La reflexión individual se entrega el miércoles 2 dic 2026, 23:59 (hora de Colombia).

| Archivo del ZIP `week-3-<group_id>.zip` | Descripción |
|-----------------------------------------|-------------|
| `week-3-group-<group_id>.ipynb` | Este cuaderno ejecutado, con la nota de la pregunta y la interpretación escritas |
| `outputs/grafo.csv` | La comunidad de cada artículo en la red, con `graph_support` |
| `outputs/prediccion.csv` | La clase que predice la red y el resultado de la predicción |

Estos archivos son el resultado de esta semana.
El cuaderno del proyecto los lee cuando arma la lista de inicio.